# Graph DAG Execution — Dependency-Aware Parallel Workflows

---

This notebook covers Multigen's `Graph` — the most powerful execution primitive. It enables
workflows where **agent execution order is determined by data dependencies**, not by a
hard-coded sequence.

By the end you will understand:
- What a DAG is and why it surpasses sequential chains for complex workflows
- How to build, run, and inspect a Graph
- Node options: retries, timeouts
- Conditional and dynamic edges
- Graph concatenation, topological sort, cycle support
- Serialising a Graph to a DSL dict for remote execution
- A 6-node machine learning inference pipeline as a real-world example

## 1. What Is a DAG? Why It Beats Sequential Chains for Complex Workflows

A **Directed Acyclic Graph (DAG)** is a graph where:
- Nodes represent computation steps (agents)
- Edges represent data dependencies ("B must run after A")
- The graph has no cycles (no step indirectly depends on itself)

### Sequential Chain limitations

```
Chain: A → B → C → D → E
Total latency = latency(A) + latency(B) + latency(C) + latency(D) + latency(E)
```

If B and C are independent (neither reads the other's output), they could run in parallel —
but a Chain forces them to run one after the other.

### DAG: run as much in parallel as the dependency structure allows

```
         ┌──────── C ───────┐
A → B → ─┤                  ├─ → E
         └──────── D ───────┘

Execution:
  Wave 1: A
  Wave 2: B  (depends on A)
  Wave 3: C and D run IN PARALLEL (both depend only on B)
  Wave 4: E  (depends on both C and D — waits for both)

Total latency = lat(A) + lat(B) + max(lat(C), lat(D)) + lat(E)
vs Chain      = lat(A) + lat(B) + lat(C) + lat(D) + lat(E)
```

The DAG automatically parallelises wherever the dependency structure allows it.
You express **what depends on what** — the scheduler handles **when to run** each node.

## 2. ASCII Diagram: The Diamond DAG

The classic test case for DAG executors is the **diamond pattern**:

```
        ┌────────────────────────────────────┐
        │         DIAMOND DAG                │
        │                                    │
        │            [fetch]                 │
        │               │                   │
        │            [parse]                 │
        │            /     \                 │
        │        [enrich_a] [enrich_b]       │
        │            \     /                 │
        │            [merge]                 │
        │                                    │
        │  Execution waves:                  │
        │    Wave 1: fetch                   │
        │    Wave 2: parse                   │
        │    Wave 3: enrich_a ∥ enrich_b     │
        │    Wave 4: merge                   │
        └────────────────────────────────────┘

enrich_a and enrich_b are independent — they can run simultaneously in Wave 3.
merge must wait for BOTH enrich_a and enrich_b to complete before starting.
```

This is impossible to express efficiently with a `Chain` — you would need a `Parallel`
step sandwiched between sequential steps, which works for one level but becomes
unmanageable for deeper or more complex dependency graphs.

In [ ]:
import asyncio
import time
import json

from multigen import Graph, GraphResult, FunctionAgent, agent

print("Multigen Graph imports successful.")

## 3. Building the Diamond DAG — `node()` and `edge()` API

A `Graph` is built by:
1. Adding **nodes** with `g.node(name, agent)` — each node wraps one agent
2. Adding **edges** with `g.edge(from_node, to_node)` — declares that `to_node` must wait
   for `from_node` to complete before starting

The graph infers execution order automatically using **topological sort**.

In [ ]:
# Define the four agents for the diamond DAG

@agent
async def fetch(ctx: dict) -> dict:
    """Simulate fetching raw data from an API (100ms)."""
    await asyncio.sleep(0.10)
    return {"raw": {"id": 42, "payload": "<xml>data</xml>", "schema": "v2"}}

@agent
async def parse(ctx: dict) -> dict:
    """Parse the raw response into structured fields (50ms)."""
    await asyncio.sleep(0.05)
    raw = ctx.get("raw", {})
    return {"parsed": {"id": raw.get("id"), "schema": raw.get("schema"), "count": 7}}

@agent
async def enrich_a(ctx: dict) -> dict:
    """Enrich with geographic data from service A (80ms)."""
    await asyncio.sleep(0.08)
    return {"geo": {"country": "US", "region": "West", "lat": 37.38, "lng": -122.08}}

@agent
async def enrich_b(ctx: dict) -> dict:
    """Enrich with user profile data from service B (120ms)."""
    await asyncio.sleep(0.12)  # slower than enrich_a
    return {"profile": {"tier": "premium", "age_days": 450, "risk_score": 0.12}}

@agent
async def merge(ctx: dict) -> dict:
    """Merge geo and profile enrichments with parsed data."""
    parsed  = ctx.get("parsed", {})
    geo     = ctx.get("geo", {})
    profile = ctx.get("profile", {})
    return {"enriched_record": {**parsed, "geo": geo, "profile": profile}}

# Build the diamond DAG
diamond = Graph(name="diamond_dag")

# Add 4 nodes
diamond.node("fetch",    fetch)
diamond.node("parse",    parse)
diamond.node("enrich_a", enrich_a)
diamond.node("enrich_b", enrich_b)
diamond.node("merge",    merge)

# Add edges (dependency declarations)
diamond.edge("fetch",    "parse")     # parse waits for fetch
diamond.edge("parse",    "enrich_a")  # enrich_a waits for parse
diamond.edge("parse",    "enrich_b")  # enrich_b waits for parse (same level → parallel!)
diamond.edge("enrich_a", "merge")     # merge waits for enrich_a
diamond.edge("enrich_b", "merge")     # merge waits for enrich_b too

print("Graph built:", diamond.name)
print("Nodes:", diamond.node_names)
print("Edges:", diamond.edge_list)

## 4. Running the Graph — `await g.run()`, Inspecting `GraphResult`

Calling `await graph.run(context)` executes the graph:
- Nodes with no unfulfilled dependencies ("ready" nodes) start immediately
- As each node completes, its output is merged into the shared context
- Newly-unblocked nodes start as soon as all their predecessors finish

The returned `GraphResult` contains:
- `result.final_output` — output of the terminal node (the node with no successors)
- `result.full_context` — the complete accumulated context
- `result.executed` — list of node names that ran, in topological order
- `result.total_ms` — wall-clock time

In [ ]:
# Run the diamond DAG
t0 = time.perf_counter()
result = await diamond.run({"request_id": "req_001"})
elapsed = (time.perf_counter() - t0) * 1000

print(f"Graph completed in {elapsed:.0f}ms")
print(f"  Expected: 100 + 50 + max(80,120) + 0 ≈ 270ms (not 100+50+80+120+0=350ms)")
print()
print("executed         :", result.executed)
# ['fetch', 'parse', 'enrich_a', 'enrich_b', 'merge'] — note: enrich_a and enrich_b ran in parallel

print("final_output     :", result.final_output)
print()
print("enriched_record  :")
record = result.full_context.get("enriched_record", {})
for k, v in record.items():
    print(f"  {k}: {v}")

## 5. Accessing Any Node's Output — `result["node_name"]`

Just like `ChainResult`, a `GraphResult` lets you access the output delta produced by
any specific node using dict-style access.

This is invaluable for debugging and for downstream code that needs to branch on
an intermediate node's output (e.g., was the geo enrichment successful?).

In [ ]:
# Access individual node outputs from the GraphResult
print("result['fetch']   :", result["fetch"])
# {'raw': {'id': 42, 'payload': '...', 'schema': 'v2'}}

print("result['parse']   :", result["parse"])
# {'parsed': {'id': 42, 'schema': 'v2', 'count': 7}}

print("result['enrich_a']:", result["enrich_a"])
# {'geo': {'country': 'US', ...}}

print("result['enrich_b']:", result["enrich_b"])
# {'profile': {'tier': 'premium', ...}}

print("result['merge']   :", result["merge"])
# {'enriched_record': {...}}

## 6. Node Options — `max_retries` and `timeout`

When adding a node to a graph, you can pass options via a third argument:

```python
g.node(
    "enrich_a",
    enrich_a_agent,
    max_retries=2,    # retry up to 2 times on failure before giving up
    timeout=15.0,     # cancel this node if it exceeds 15 seconds
    retry_delay=0.5,  # wait 500ms between retries
)
```

### Why per-node options?

Different nodes have different reliability profiles:
- A fast local computation: `timeout=0.1, max_retries=0`
- A flaky external API: `timeout=10.0, max_retries=3`
- An LLM call (rate-limited): `timeout=60.0, max_retries=2, retry_delay=5.0`

Per-node configuration lets you tune each node independently.

In [ ]:
# Track retry attempts to verify max_retries is working
attempt_counter = {"flaky_service": 0}

@agent
async def stable_fetch(ctx: dict) -> dict:
    """A stable node that always succeeds."""
    await asyncio.sleep(0.02)
    return {"raw_data": "document_payload"}

@agent
async def flaky_service(ctx: dict) -> dict:
    """Fails on first 2 attempts, succeeds on the 3rd."""
    attempt_counter["flaky_service"] += 1
    attempt = attempt_counter["flaky_service"]
    if attempt < 3:
        raise ConnectionError(f"Service unavailable (attempt {attempt})")
    return {"enriched": True, "attempt": attempt}

@agent
async def finalise(ctx: dict) -> dict:
    return {"done": True, "enriched": ctx.get("enriched", False)}

# Graph with retry config on the flaky node
retry_graph = Graph(name="retry_demo")
retry_graph.node("stable_fetch",  stable_fetch)
retry_graph.node(
    "flaky_service",
    flaky_service,
    max_retries=3,     # allow up to 3 retries (4 total attempts)
    retry_delay=0.01,  # 10ms between retries (fast for demo)
)
retry_graph.node("finalise", finalise)

retry_graph.edge("stable_fetch", "flaky_service")
retry_graph.edge("flaky_service", "finalise")

attempt_counter["flaky_service"] = 0  # reset
result = await retry_graph.run({})

print("Graph succeeded :", result.succeeded)
print("Total attempts  :", attempt_counter["flaky_service"])  # 3
print("flaky_service   :", result["flaky_service"])            # {'enriched': True, 'attempt': 3}

## 7. Conditional Edges — `condition=lambda ctx: ...`

An edge can carry a **condition** — a predicate that is evaluated when the source node
completes. If the condition returns `False`, the edge is not traversed, meaning the
destination node is not scheduled (unless it has another satisfied incoming edge).

This lets you express branching logic directly in the graph topology:

```
parse ──(count > 0)──> enrich_a
parse ──(always)────> log_empty
```

If `parse` produces `count=0`, the edge to `enrich_a` is not taken and `enrich_a` is skipped.

In [ ]:
@agent
async def validate(ctx: dict) -> dict:
    """Validate input and set 'valid' flag."""
    text = ctx.get("text", "")
    return {"valid": len(text.strip()) > 0, "char_count": len(text.strip())}

@agent
async def run_expensive_analysis(ctx: dict) -> dict:
    """Only runs on valid, non-empty input."""
    await asyncio.sleep(0.05)
    return {"analysis": f"Processed {ctx.get('char_count', 0)} chars", "ran": True}

@agent
async def handle_empty(ctx: dict) -> dict:
    """Fallback for empty input."""
    return {"analysis": "Empty input — nothing to process", "ran": False}

@agent
async def emit_result(ctx: dict) -> dict:
    """Always runs — combines analysis output regardless of which branch ran."""
    return {"final": ctx.get("analysis", "unknown")}

# Build the conditional graph
conditional_graph = Graph(name="conditional_routing")
conditional_graph.node("validate",              validate)
conditional_graph.node("run_expensive_analysis", run_expensive_analysis)
conditional_graph.node("handle_empty",           handle_empty)
conditional_graph.node("emit_result",            emit_result)

# Conditional edge: only go to expensive analysis if input is valid
conditional_graph.edge(
    "validate",
    "run_expensive_analysis",
    condition=lambda ctx: ctx.get("valid", False)
)
# Conditional edge: only go to empty handler if input is NOT valid
conditional_graph.edge(
    "validate",
    "handle_empty",
    condition=lambda ctx: not ctx.get("valid", False)
)
# Both branches converge at emit_result
conditional_graph.edge("run_expensive_analysis", "emit_result")
conditional_graph.edge("handle_empty",           "emit_result")

# Test with valid input
r1 = await conditional_graph.run({"text": "Hello, meaningful text here."})
print("Valid input   — executed:", r1.executed)
print("               final   :", r1.full_context["final"])
print()

# Test with empty input
r2 = await conditional_graph.run({"text": "   "})
print("Empty input   — executed:", r2.executed)
print("               final   :", r2.full_context["final"])

## 8. Dynamic Routing — `dynamic=True` Edge

A **dynamic edge** means the source agent's output **determines the next node** at runtime.
The agent returns a special key (e.g., `"__next__"`) containing the name of the node to
execute next. The Graph executor reads this key and routes accordingly.

This is more powerful than conditional edges because the routing decision can be based on
complex logic computed inside the agent — including LLM-generated decisions.

```python
@agent
async def router(ctx: dict) -> dict:
    # LLM or rule-based decision
    next_node = "fast_path" if ctx["size"] < 1000 else "slow_path"
    return {"__next__": next_node}

g.edge("router", "fast_path", dynamic=True)
g.edge("router", "slow_path", dynamic=True)
```

In [ ]:
@agent
async def ingest(ctx: dict) -> dict:
    """Ingest a document and determine its size category."""
    text = ctx.get("text", "")
    return {"text": text, "word_count": len(text.split())}

@agent
async def size_router(ctx: dict) -> dict:
    """Dynamically route based on document size."""
    wc = ctx.get("word_count", 0)
    if wc < 50:
        next_node = "process_short"
    elif wc < 200:
        next_node = "process_medium"
    else:
        next_node = "process_long"
    print(f"  [size_router] word_count={wc} → routing to '{next_node}'")
    return {"__next__": next_node, "route_taken": next_node}

@agent
async def process_short(ctx: dict) -> dict:
    return {"result": "SHORT pipeline applied", "pipeline": "short"}

@agent
async def process_medium(ctx: dict) -> dict:
    await asyncio.sleep(0.02)
    return {"result": "MEDIUM pipeline applied", "pipeline": "medium"}

@agent
async def process_long(ctx: dict) -> dict:
    await asyncio.sleep(0.05)
    return {"result": "LONG pipeline applied", "pipeline": "long"}

dynamic_graph = Graph(name="dynamic_routing")
dynamic_graph.node("ingest",         ingest)
dynamic_graph.node("size_router",    size_router)
dynamic_graph.node("process_short",  process_short)
dynamic_graph.node("process_medium", process_medium)
dynamic_graph.node("process_long",   process_long)

dynamic_graph.edge("ingest",      "size_router")
# Dynamic edges: size_router's output determines which one activates
dynamic_graph.edge("size_router", "process_short",  dynamic=True)
dynamic_graph.edge("size_router", "process_medium", dynamic=True)
dynamic_graph.edge("size_router", "process_long",   dynamic=True)

# Test with three different document sizes
for doc_text, label in [
    ("word " * 10, "10 words (short)"),
    ("word " * 100, "100 words (medium)"),
    ("word " * 300, "300 words (long)"),
]:
    r = await dynamic_graph.run({"text": doc_text.strip()})
    print(f"  {label:22s} → pipeline={r.full_context['pipeline']}")

## 9. Graph `>>` Graph Concatenation

The `>>` operator concatenates two graphs: the terminal nodes of the left graph are
connected to the entry nodes of the right graph. All nodes and edges are merged
into a new combined graph.

This is the graph-level equivalent of `chain_a | chain_b` and lets you build
reusable sub-graphs that can be composed into larger pipelines.

In [ ]:
# Sub-graph 1: ingestion (fetch + validate)
@agent
async def raw_fetch(ctx: dict) -> dict:
    await asyncio.sleep(0.03)
    return {"raw": "<data>payload</data>"}

@agent
async def schema_validate(ctx: dict) -> dict:
    raw = ctx.get("raw", "")
    return {"validated": True, "payload": raw.replace("<data>", "").replace("</data>", "")}

ingest_graph = Graph(name="ingest_subgraph")
ingest_graph.node("raw_fetch",       raw_fetch)
ingest_graph.node("schema_validate", schema_validate)
ingest_graph.edge("raw_fetch", "schema_validate")

# Sub-graph 2: transformation (transform + emit)
@agent
async def transform_payload(ctx: dict) -> dict:
    payload = ctx.get("payload", "")
    return {"transformed": payload.strip().upper()}

@agent
async def emit_to_sink(ctx: dict) -> dict:
    return {"emitted": True, "final_value": ctx.get("transformed", "")}

transform_graph = Graph(name="transform_subgraph")
transform_graph.node("transform_payload", transform_payload)
transform_graph.node("emit_to_sink",      emit_to_sink)
transform_graph.edge("transform_payload", "emit_to_sink")

# Concatenate using >> operator
full_pipeline_graph = ingest_graph >> transform_graph

print("Combined graph nodes:", full_pipeline_graph.node_names)
# ['raw_fetch', 'schema_validate', 'transform_payload', 'emit_to_sink']

r = await full_pipeline_graph.run({})
print("Executed:", r.executed)
print("Final value:", r.full_context["final_value"])

## 10. Topological Sort Visualisation — Print the Execution Order

Before running a graph, it can be useful to **visualise the topological sort** — the order
in which nodes will execute and which nodes run in parallel (the same "wave").

The `graph.topological_waves()` method returns a list of lists, where each inner list is
one execution wave (all nodes in a wave can run concurrently).

In [ ]:
# Print execution waves for the diamond DAG
def print_execution_plan(g: Graph):
    """Print a visual execution plan showing parallel waves."""
    print(f"Execution plan for '{g.name}':")
    print("─" * 50)
    waves = g.topological_waves()
    for i, wave in enumerate(waves):
        parallel_indicator = " ∥ " if len(wave) > 1 else "   "
        nodes_str = parallel_indicator.join(f"[{n}]" for n in wave)
        label = "(parallel)" if len(wave) > 1 else ""
        print(f"  Wave {i+1}: {nodes_str}  {label}")
    print("─" * 50)
    total_nodes = sum(len(w) for w in waves)
    print(f"  {total_nodes} nodes across {len(waves)} waves")

print_execution_plan(diamond)
print()
print_execution_plan(full_pipeline_graph)

## 11. Cycle Support — `is_loop=True` with a Loop Counter

By default, Graphs are acyclic (DAG). But Multigen supports **controlled loops** via
`is_loop=True` on an edge. A loop edge connects a downstream node back to an upstream
one, creating a cycle. A `max_iterations` parameter prevents infinite loops.

Use cases:
- **Iterative refinement**: an LLM generates output, a validator checks it, and if it
  fails validation the output is sent back to the LLM for correction
- **Active learning**: process a batch, update a model, check convergence, repeat
- **Retry with context**: unlike `max_retries`, the loop can modify the context before
  re-running the node (e.g., adding error feedback to the LLM prompt)

```python
g.edge("validate", "generate", is_loop=True, condition=lambda ctx: not ctx["valid"])
```

In [ ]:
# Iterative refinement loop:
# generate → validate → (if invalid) → generate (loop) → (if valid) → finalise

iteration_log = []

@agent
async def generate_solution(ctx: dict) -> dict:
    """Simulate a generative step that improves with each iteration."""
    iteration = ctx.get("__loop_count__", 0)
    iteration_log.append(f"generate iteration {iteration}")
    # Quality improves each iteration (mock)
    quality = min(0.3 + iteration * 0.4, 1.0)
    return {"output": f"solution_v{iteration}", "quality": quality}

@agent
async def validate_solution(ctx: dict) -> dict:
    """Accept the solution only when quality >= 0.9."""
    quality = ctx.get("quality", 0)
    accepted = quality >= 0.9
    return {"accepted": accepted, "quality": quality}

@agent
async def finalise_solution(ctx: dict) -> dict:
    """Store the accepted solution."""
    return {"final_solution": ctx.get("output"), "final_quality": ctx.get("quality")}

loop_graph = Graph(name="refinement_loop", max_iterations=10)
loop_graph.node("generate_solution",  generate_solution)
loop_graph.node("validate_solution",  validate_solution)
loop_graph.node("finalise_solution",  finalise_solution)

loop_graph.edge("generate_solution", "validate_solution")
# Loop back to generate if not accepted yet
loop_graph.edge(
    "validate_solution", "generate_solution",
    is_loop=True,
    condition=lambda ctx: not ctx.get("accepted", False)
)
# Proceed to finalise when accepted
loop_graph.edge(
    "validate_solution", "finalise_solution",
    condition=lambda ctx: ctx.get("accepted", False)
)

iteration_log.clear()
r = await loop_graph.run({})

print("Iteration log:", iteration_log)
print("Final solution:", r.full_context.get("final_solution"))
print("Final quality :", r.full_context.get("final_quality"))

## 12. Converting to DSL Dict — `g.to_dict()` for Remote Execution

A Graph can be serialised to a plain Python dict using `g.to_dict()`. This dict follows
the Multigen DSL format and can be:
- Sent over HTTP to a Multigen server for remote execution
- Stored as JSON in a database
- Versioned in source control as a workflow definition
- Reconstructed into a Graph with `Graph.from_dict(d)`

The DSL dict captures the graph **topology** (nodes, edges, options) but not the
agent implementations — those are referenced by name and resolved on the execution server.

In [ ]:
# Serialise the diamond DAG to a dict
dsl = diamond.to_dict()

print("DSL dict structure:")
print(json.dumps(dsl, indent=2, default=str)[:2000])  # first 2000 chars

# The DSL includes:
# - 'name': graph name
# - 'nodes': list of {name, agent_ref, options}
# - 'edges': list of {from, to, condition_expr, dynamic, is_loop}

In [ ]:
# Reconstruct the graph from the DSL dict
# In production, the server has the agent implementations registered by name.
# Locally, we pass an agent registry dict.

agent_registry = {
    "fetch":    fetch,
    "parse":    parse,
    "enrich_a": enrich_a,
    "enrich_b": enrich_b,
    "merge":    merge,
}

reconstructed = Graph.from_dict(dsl, agent_registry=agent_registry)
print("Reconstructed graph nodes:", reconstructed.node_names)

# Run the reconstructed graph — should produce identical results
r = await reconstructed.run({"request_id": "req_round_trip"})
print("Reconstructed graph final output:", r.final_output)

## 13. Parallel Wave Visualisation — Which Nodes Run in the Same Wave

Let's build a more complex graph and visualise its parallel wave structure.
This helps you understand and optimise the theoretical minimum latency of your workflow.

In [ ]:
# Build a more complex DAG to visualise
def make_node(name: str, sleep_ms: int) -> FunctionAgent:
    async def fn(ctx: dict) -> dict:
        await asyncio.sleep(sleep_ms / 1000)
        return {name: f"done_{sleep_ms}ms"}
    return FunctionAgent(fn=fn, name=name)

complex_g = Graph(name="complex_dag")
# Latencies in ms
node_configs = [
    ("A", 50), ("B", 80), ("C", 30), ("D", 60),
    ("E", 40), ("F", 20), ("G", 70), ("H", 90),
]
for name, ms in node_configs:
    complex_g.node(name, make_node(name, ms))

# Dependency structure:
# A and B are independent (wave 1)
# C depends on A; D depends on A (wave 2, parallel)
# E depends on B; F depends on B (wave 2, parallel)
# G depends on C and E (wave 3)
# H depends on D, F, and G (wave 4)
complex_g.edge("A", "C"); complex_g.edge("A", "D")
complex_g.edge("B", "E"); complex_g.edge("B", "F")
complex_g.edge("C", "G"); complex_g.edge("E", "G")
complex_g.edge("D", "H"); complex_g.edge("F", "H"); complex_g.edge("G", "H")

print_execution_plan(complex_g)
print()

# Calculate theoretical minimum latency
latencies = dict(node_configs)
waves = complex_g.topological_waves()
total_theoretical = sum(max(latencies[n] for n in wave) for wave in waves)
sequential_total = sum(latencies.values())

print(f"Sequential latency  : {sequential_total}ms")
print(f"Parallel (DAG) latency: {total_theoretical}ms")
print(f"Speedup: {sequential_total/total_theoretical:.2f}x")

# Verify empirically
t0 = time.perf_counter()
r = await complex_g.run({})
actual = (time.perf_counter() - t0) * 1000
print(f"Actual wall time    : {actual:.0f}ms")

## 14. Real-World Example: 6-Node ML Inference Pipeline

A full machine learning inference pipeline with parallel feature extraction:

```
        [ingest]          ← Load raw input (text + metadata)
           │
        [validate]        ← Schema validation + sanitisation
        /   |   \
[feat_A] [feat_B] [feat_C]  ← Three independent feature extractors (parallel)
        \   |   /
        [model]           ← Combine features, run inference
           │
        [predict]         ← Format and return prediction
```

This represents a realistic ML serving pipeline where:
- Feature extraction from 3 sources (text, image, tabular) runs in parallel
- The model node waits for all three feature vectors
- The predict node post-processes the model output

In [ ]:
# Node 1: Ingest raw input
@agent
async def ingest_input(ctx: dict) -> dict:
    """Load and deserialise the incoming request payload."""
    await asyncio.sleep(0.01)  # 10ms
    raw = ctx.get("raw_input", {})
    return {
        "text": raw.get("text", "default input text"),
        "tabular": raw.get("tabular", {"age": 35, "income": 60000}),
        "image_url": raw.get("image_url", "https://example.com/img.jpg"),
    }

# Node 2: Validate
@agent
async def validate_input(ctx: dict) -> dict:
    """Validate all required fields are present and well-formed."""
    missing = []
    if not ctx.get("text"):       missing.append("text")
    if not ctx.get("tabular"):    missing.append("tabular")
    if not ctx.get("image_url"): missing.append("image_url")
    return {"validation_passed": len(missing) == 0, "missing_fields": missing}

# Node 3a: Text feature extraction
@agent
async def extract_text_features(ctx: dict) -> dict:
    """Convert text to embedding vector (mocked, 80ms)."""
    await asyncio.sleep(0.08)
    text = ctx.get("text", "")
    # Mock 8-dimensional embedding
    embedding = [hash(w) % 100 / 100 for w in text.split()[:8]]
    while len(embedding) < 8:
        embedding.append(0.0)
    return {"text_features": embedding[:8]}

# Node 3b: Tabular feature extraction
@agent
async def extract_tabular_features(ctx: dict) -> dict:
    """Normalise and encode tabular fields (40ms)."""
    await asyncio.sleep(0.04)
    tab = ctx.get("tabular", {})
    # Mock normalisation
    age_norm    = tab.get("age", 30) / 100.0
    income_norm = tab.get("income", 50000) / 200000.0
    return {"tabular_features": [age_norm, income_norm]}

# Node 3c: Image feature extraction
@agent
async def extract_image_features(ctx: dict) -> dict:
    """Fetch and embed image (mocked, 120ms — slowest path)."""
    await asyncio.sleep(0.12)
    url = ctx.get("image_url", "")
    # Mock 4-dimensional CNN embedding
    mock_embedding = [0.31, 0.72, 0.18, 0.95]
    return {"image_features": mock_embedding}

# Node 4: Model inference
@agent
async def run_model(ctx: dict) -> dict:
    """Concatenate feature vectors and run the classifier (30ms)."""
    await asyncio.sleep(0.03)
    # Concatenate all feature vectors
    features = (
        ctx.get("text_features", []) +
        ctx.get("tabular_features", []) +
        ctx.get("image_features", [])
    )
    # Mock model: compute weighted sum as score
    score = sum(f * (i + 1) for i, f in enumerate(features)) / len(features) if features else 0
    return {"model_score": round(score, 4), "feature_dim": len(features)}

# Node 5: Post-process prediction
@agent
async def format_prediction(ctx: dict) -> dict:
    """Convert raw model score into a human-readable prediction."""
    score = ctx.get("model_score", 0.5)
    label = "approved" if score > 0.5 else "declined"
    confidence = abs(score - 0.5) * 2  # 0-1 scale
    return {
        "prediction": {
            "label": label,
            "score": score,
            "confidence": round(confidence, 3),
            "feature_dim": ctx.get("feature_dim", 0),
        }
    }

# Assemble the 6-node ML pipeline graph
ml_graph = Graph(name="ml_inference_pipeline")

ml_graph.node("ingest",          ingest_input)
ml_graph.node("validate",        validate_input,         max_retries=0, timeout=2.0)
ml_graph.node("feat_text",       extract_text_features,  timeout=5.0)
ml_graph.node("feat_tabular",    extract_tabular_features, timeout=2.0)
ml_graph.node("feat_image",      extract_image_features,  timeout=10.0)
ml_graph.node("model",           run_model,              timeout=30.0)
ml_graph.node("predict",         format_prediction)

# Edges
ml_graph.edge("ingest",       "validate")
# Three parallel feature extraction nodes (all wait for validate)
ml_graph.edge("validate",     "feat_text",    condition=lambda ctx: ctx.get("validation_passed"))
ml_graph.edge("validate",     "feat_tabular", condition=lambda ctx: ctx.get("validation_passed"))
ml_graph.edge("validate",     "feat_image",   condition=lambda ctx: ctx.get("validation_passed"))
# Model waits for all three feature extractors
ml_graph.edge("feat_text",    "model")
ml_graph.edge("feat_tabular", "model")
ml_graph.edge("feat_image",   "model")
ml_graph.edge("model",        "predict")

print("ML pipeline graph built.")
print_execution_plan(ml_graph)

In [ ]:
# Run the ML inference pipeline
sample_request = {
    "raw_input": {
        "text": "Customer has an excellent credit history and stable employment.",
        "tabular": {"age": 42, "income": 95000},
        "image_url": "https://example.com/selfie.jpg",
    }
}

t0 = time.perf_counter()
result = await ml_graph.run(sample_request)
elapsed = (time.perf_counter() - t0) * 1000

print(f"ML pipeline completed in {elapsed:.0f}ms")
print(f"  Sequential would take: {10+10+80+40+120+30}ms = 290ms")
print(f"  Parallel (waves): 10+10+max(80,40,120)+30 = ~240ms")
print()
print("Execution order:", result.executed)
print()
print("Validation result :", result["validate"])
print("Text features dim :", len(result.full_context.get("text_features", [])))
print("Model score       :", result.full_context.get("model_score"))
print("Feature dimensions:", result.full_context.get("feature_dim"))
print()
pred = result.full_context.get("prediction", {})
print("═" * 40)
print(f"PREDICTION: {pred.get('label', '?').upper()}")
print(f"  Score     : {pred.get('score'):.4f}")
print(f"  Confidence: {pred.get('confidence'):.3f}")
print("═" * 40)

In [ ]:
# Test with invalid input to verify conditional edges skip feature extraction
invalid_request = {
    "raw_input": {}  # empty — validation will fail
}

r_invalid = await ml_graph.run(invalid_request)
print("Invalid input run:")
print("  executed        :", r_invalid.executed)
print("  validation_passed:", r_invalid.full_context.get("validation_passed"))
print("  missing_fields  :", r_invalid.full_context.get("missing_fields"))
# Feature extractors and model should NOT be in executed — conditional edges prevented them

## Summary

The `Graph` is Multigen's most expressive execution primitive. Its key advantages:

| Feature | API |
|---|---|
| Add a node | `g.node(name, agent, timeout=..., max_retries=...)` |
| Add an edge | `g.edge(from, to, condition=..., dynamic=..., is_loop=...)` |
| Run the graph | `await g.run(ctx)` |
| Access any node output | `result["node_name"]` |
| See execution order | `result.executed` |
| Visualise parallel waves | `g.topological_waves()` |
| Concatenate graphs | `graph_a >> graph_b` |
| Serialise to DSL | `g.to_dict()` / `Graph.from_dict(d, registry)` |
| Loops | `g.edge(..., is_loop=True, condition=...)` |
| Dynamic routing | Agent returns `{"__next__": node_name}`, edge has `dynamic=True` |

### When to choose Graph over Chain

- Use `Chain` when steps are **strictly sequential** (each needs the previous step's output)
- Use `Graph` when steps have **partial dependencies** (some can run in parallel)
- Use `Graph` when you need **conditional branching** at the topology level
- Use `Graph` when you need **iterative loops** with context-aware termination
- Use `Graph` when you want **workflow portability** via DSL serialisation

You have now completed all four local SDK notebooks. The full learning path:

```
12_local_sdk_agents.ipynb      ← Agent primitives
13_sequential_chains.ipynb     ← Sequential composition
14_parallel_execution.ipynb    ← Parallel patterns
15_graph_dag_local.ipynb       ← Dependency-aware DAG execution  ← you are here
```